# 4. Vanna Training

This notebook trains Vanna with the schema and sample queries for the EMR database.

In [ ]:
import os
import sys

sys.path.append(os.path.abspath('..'))
from src.utils import get_vanna

In [ ]:
print("1. Initializing Vanna...")
# Ensure PostgreSQL is running and notebook 3 has been executed
vn = get_vanna()

In [ ]:
print("2. Training Vanna with DDL...")
ddl = """
CREATE TABLE emr_records (
    emr_name VARCHAR,
    machine_model VARCHAR,
    model_family VARCHAR,
    serial_number VARCHAR,
    branch_site VARCHAR,
    account VARCHAR,
    sub_call_type VARCHAR,
    pmact_type VARCHAR,
    subjects TEXT,
    symptom TEXT,
    caused_of_problem TEXT,
    techcare_component VARCHAR,
    techcare_sub_component VARCHAR,
    main_cause_part_no VARCHAR,
    part_description VARCHAR,
    created_date TIMESTAMP,
    closed_date TIMESTAMP,
    cluster_id INTEGER,
    cluster_prob FLOAT,
    cluster_label VARCHAR
);
"""
vn.train(ddl=ddl)

In [ ]:
print("3. Training Vanna with Documentation...")
doc = """
The 'emr_records' table contains Equipment Maintenance Records for heavy machinery.
- 'machine_model' is the specific equipment model (e.g., PC200-8).
- 'model_family' is the broader category (e.g., PC200).
- 'branch_site' is the location where the equipment is operated.
- 'cluster_label' categorizes the text description of the fault into a common type (e.g., 'Hydraulic Leak').
- Use 'created_date' when filtering by month or year.
"""
vn.train(documentation=doc)

In [ ]:
print("4. Training Vanna with Sample Q&A...")

qa_pairs = [
    ("Berapa total fault per model?", "SELECT machine_model, COUNT(*) as total FROM emr_records GROUP BY machine_model ORDER BY total DESC;"),
    ("Model apa yang paling sering rusak?", "SELECT machine_model, COUNT(*) as total FROM emr_records GROUP BY machine_model ORDER BY total DESC LIMIT 1;"),
    ("Tampilkan 5 site dengan masalah terbanyak", "SELECT branch_site, COUNT(*) as total FROM emr_records GROUP BY branch_site ORDER BY total DESC LIMIT 5;"),
    ("Berapa jumlah kerusakan berdasarkan kategori (cluster)?", "SELECT cluster_label, COUNT(*) as total FROM emr_records GROUP BY cluster_label ORDER BY total DESC;"),
    ("Tampilkan tren kerusakan per bulan", "SELECT DATE_TRUNC('month', created_date) as month, COUNT(*) as total FROM emr_records GROUP BY month ORDER BY month;"),
    ("Tampilkan tren masalah pada PC200 per bulan", "SELECT DATE_TRUNC('month', created_date) as month, COUNT(*) as total FROM emr_records WHERE model_family = 'PC200' GROUP BY month ORDER BY month;"),
    ("Sebutkan top 3 masalah (cluster) di site Jakarta", "SELECT cluster_label, COUNT(*) as total FROM emr_records WHERE branch_site ILIKE '%Jakarta%' GROUP BY cluster_label ORDER BY total DESC LIMIT 3;")
]

for q, sql in qa_pairs:
    vn.train(question=q, sql=sql)

print("Training complete.")

In [ ]:
print("5. Testing SQL Generation...")
# This will use the local Ollama LLM defined in our custom Vanna implementation
test_query = "Berapa banyak kerusakan hydraulic pada PC200?"
sql = vn.generate_sql(test_query)
print(f"Question: {test_query}\nGenerated SQL: {sql}")

if sql:
    df = vn.run_sql(sql)
    print(df)